# 02. The Agent Loop

## What you'll learn

- **What makes something an agent** — the difference between a single LLM call and an autonomous loop
- **The basic agent loop** — `while True` with an LLM deciding what happens at each step
- **Stop conditions** — max iterations, explicit stop tokens, and why safety limits matter
- **Conversation history as working memory** — how appending messages lets the agent "think" across steps
- **Building agents with distinct step-by-step behavior** — controlling the LLM's output at each iteration

## Prerequisites

- [01. Hello LLM](01_hello_llm.ipynb) — making your first API call with `chat()`
- [Appendix 01. Python Fundamentals](../appendix/01_python_fundamentals.ipynb) — variables, loops, control flow
- [Appendix 04. Strings and JSON](../appendix/04_strings_and_json.ipynb) — string parsing and manipulation
- [Appendix 05. Error Handling](../appendix/05_error_handling.ipynb) — `try`/`except` and retry patterns

## Why this matters

A chatbot makes one LLM call and returns the answer. An **agent** makes *many* calls in a loop, deciding at each step whether to keep going or stop. This is the single most important pattern in all of agentic AI: the **agent loop**. Every agent framework — LangChain, CrewAI, smolagents, AutoGPT — is ultimately a dressed-up version of the loop you'll build in this notebook from scratch.

> **Further reading:**
> - [LLM Powered Autonomous Agents (Lilian Weng)](https://lilianweng.github.io/posts/2023-06-23-agent/) — the definitive overview of agent architectures, memory, planning, and tool use
> - [Building effective agents (Anthropic)](https://docs.anthropic.com/en/docs/build-with-claude/agentic) — practical patterns for agent loops and orchestration

---
## 1. What Makes an Agent?

An agent isn't a single LLM call. It's a **loop** where the LLM makes decisions at each step. The key difference:

| | Chatbot | Agent |
|---|---|---|
| LLM calls | One | Many (in a loop) |
| Memory | None between calls | Conversation history grows each step |
| Who decides when to stop? | The developer (fixed) | The LLM (dynamic) |
| Can it do multi-step reasoning? | No | Yes |

Here's the fundamental pattern:

```
┌─────────────────────────────────────┐
│           AGENT LOOP                │
│                                     │
│   ┌──────────┐                      │
│   │  Prompt   │◄──── User query     │
│   └────┬─────┘                      │
│        │                            │
│        ▼                            │
│   ┌──────────┐                      │
│   │ Call LLM  │                     │
│   └────┬─────┘                      │
│        │                            │
│        ▼                            │
│   ┌──────────┐    Yes    ┌────────┐ │
│   │  Done?    │─────────►│ Return │ │
│   └────┬─────┘          └────────┘ │
│        │ No                         │
│        │                            │
│        ▼                            │
│   ┌──────────────┐                  │
│   │ Append to     │                 │
│   │ message history│                │
│   └──────┬───────┘                  │
│          │                          │
│          └──────────► (back to top) │
└─────────────────────────────────────┘
```

The LLM **decides** when the task is done by emitting a special stop token (like `FINAL ANSWER:`). Until then, the loop keeps running, and the LLM sees its own prior reasoning through the growing message history.

---
## 2. The Simplest Agent Loop

Let's build the most minimal agent possible. The protocol is simple:

1. We give the LLM a system prompt that explains the loop rules
2. We send a question
3. The LLM responds — if its response contains `FINAL ANSWER:`, we extract the answer and stop
4. Otherwise, we add its response to the history and call it again

The system prompt *is* the protocol. It tells the LLM: "Think step by step. When you have a final answer, respond with `FINAL ANSWER:` followed by your answer."

In [ ]:
import sys
import time

sys.path.insert(0, "..")
from utils import chat

In [ ]:
SYSTEM_PROMPT = """You are a helpful assistant. Think step by step.
When you have a final answer, respond with FINAL ANSWER: followed by your answer.
Do not include FINAL ANSWER: until you are ready to give your complete response."""

question = "What are the 3 primary colors? List them one by one."

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": question},
]

print(f"Question: {question}\n")
print("=" * 50)

step = 0

while True:
    step += 1
    print(f"\n--- Step {step} ---")

    try:
        response = chat(messages, temperature=0.3)
    except RuntimeError as e:
        print(f"LLM call failed: {e}")
        break

    print(f"LLM: {response}")

    # Check for the stop condition
    if "FINAL ANSWER:" in response:
        final_answer = response.split("FINAL ANSWER:")[1].strip()
        print("\n" + "=" * 50)
        print(f"Agent finished in {step} step(s).")
        print(f"Final answer: {final_answer}")
        break

    # Not done yet — add response to history and continue
    messages.append({"role": "assistant", "content": response})
    messages.append({"role": "user", "content": "Continue."})

    time.sleep(1)

That's it. That's an agent. A `while True` loop where the LLM decides when to stop.

Notice a few things:
- We check for `"FINAL ANSWER:"` in the response string to detect the stop condition
- When the LLM isn't done, we append its response as an `assistant` message and add a `user` message saying `"Continue."` to prompt the next step
- The message list grows with each iteration — that's how the LLM "remembers" what it already said

But there's a problem with this loop: **what if the LLM never says `FINAL ANSWER:`?** The loop runs forever. Let's fix that.

---
## 3. Stop Conditions

Every agent loop needs to stop eventually. There are three types of stop conditions:

1. **Explicit stop token** — The LLM outputs a special string like `FINAL ANSWER:` (the approach we used above)
2. **Max iterations** — A hard cap on how many steps the agent can take (the safety net)
3. **Task completion** — The agent detects that a specific condition is met (e.g., all sub-tasks done)

### Why max iterations matters

Without a `max_steps` limit, a confused LLM can loop indefinitely:
- It might never produce the stop token
- It might get stuck in a reasoning loop ("Let me think more about this..." forever)
- Each iteration costs API tokens (and money)

**Always** include a max iterations guard. Here's the pattern with all three stop conditions:

In [ ]:
SYSTEM_PROMPT = """You are a helpful assistant. Think step by step.
When you have a final answer, respond with FINAL ANSWER: followed by your answer.
Do not include FINAL ANSWER: until you are ready to give your complete response."""


def run_agent(question: str, max_steps: int = 5) -> str:
    """Run a simple agent loop with a safety limit.

    Args:
        question: The question to answer.
        max_steps: Maximum number of LLM calls before forcing a stop.

    Returns:
        The agent's final answer, or a timeout message.
    """
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]

    for step in range(1, max_steps + 1):
        print(f"\n--- Step {step}/{max_steps} ---")

        try:
            response = chat(messages, temperature=0.3)
        except RuntimeError as e:
            print(f"LLM call failed: {e}")
            return f"Error: {e}"

        print(f"LLM: {response[:200]}{'...' if len(response) > 200 else ''}")

        # Stop condition 1: explicit stop token
        if "FINAL ANSWER:" in response:
            final_answer = response.split("FINAL ANSWER:")[1].strip()
            print(f"\nAgent finished in {step} step(s).")
            return final_answer

        # Not done — add to history and continue
        messages.append({"role": "assistant", "content": response})
        messages.append({"role": "user", "content": "Continue."})

        time.sleep(1)

    # Stop condition 2: max iterations reached
    print(f"\nAgent hit max steps ({max_steps}) without finishing.")
    return "Agent timed out — no final answer produced."

In [ ]:
# Test it — should finish well within 5 steps
answer = run_agent("What are the 3 primary colors? List them one by one.", max_steps=5)
print(f"\nAnswer: {answer}")

In [ ]:
# Test the safety limit — set max_steps very low to see the timeout
answer = run_agent(
    "Write a detailed 10-paragraph essay about the history of computing.",
    max_steps=2,
)
print(f"\nAnswer: {answer}")

Notice the difference between the two calls:
- The simple question finishes with `FINAL ANSWER:` within the step limit
- The hard question hits the `max_steps` ceiling and returns a timeout message

Using `for step in range(1, max_steps + 1)` instead of `while True` makes the safety limit automatic — the loop physically cannot exceed `max_steps` iterations.

---
## 4. Conversation History as Working Memory

The `messages` list is the agent's **working memory**. Each iteration, we append the LLM's response and a follow-up prompt. When the LLM is called again, it sees everything it said before — and can build on its prior reasoning.

Let's make this visible by printing the full message history at each step.

In [ ]:
SYSTEM_PROMPT = """You are a helpful assistant. Think step by step.
When you have a final answer, respond with FINAL ANSWER: followed by your answer.
Do not include FINAL ANSWER: until you are ready to give your complete response."""

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": "What is 15 * 7? Work through it step by step."},
]

max_steps = 5

for step in range(1, max_steps + 1):
    print(f"\n{'=' * 50}")
    print(f"STEP {step} — Messages in history: {len(messages)}")
    print(f"{'=' * 50}")

    # Show what the LLM will see
    for i, msg in enumerate(messages):
        role = msg['role'].upper()
        content = msg['content'][:80] + ('...' if len(msg['content']) > 80 else '')
        print(f"  [{i}] {role}: {content}")

    try:
        response = chat(messages, temperature=0.3)
    except RuntimeError as e:
        print(f"LLM call failed: {e}")
        break

    print(f"\n  LLM response: {response}")

    if "FINAL ANSWER:" in response:
        final_answer = response.split("FINAL ANSWER:")[1].strip()
        print(f"\nDone! Final answer: {final_answer}")
        break

    messages.append({"role": "assistant", "content": response})
    messages.append({"role": "user", "content": "Continue."})

    time.sleep(1)

Watch how the message count grows: 2, 4, 6, 8... Each iteration adds two messages (assistant response + user follow-up). The LLM sees its entire chain of reasoning, which is how it can build multi-step solutions.

### What happens without history?

Let's see what happens when the agent has **no memory** — we reset the messages each time instead of appending. The agent can't remember what it already did.

In [ ]:
# Agent WITHOUT memory — messages reset each step
SYSTEM_PROMPT = """You are a helpful assistant. Think step by step.
When you have a final answer, respond with FINAL ANSWER: followed by your answer.
Do not include FINAL ANSWER: until you are ready to give your complete response."""

question = "Count from 1 to 3 one number at a time. Only say the next number."

max_steps = 5

print("AGENT WITHOUT MEMORY")
print("(Messages are reset each step — the LLM can't see prior responses)\n")

for step in range(1, max_steps + 1):
    # BUG: we recreate messages from scratch every time!
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]

    try:
        response = chat(messages, temperature=0.3)
    except RuntimeError as e:
        print(f"LLM call failed: {e}")
        break

    print(f"Step {step}: {response[:100]}")

    if "FINAL ANSWER:" in response:
        break

    time.sleep(1)

print("\nNotice: the agent keeps giving the same first response because it")
print("can't see what it said before. Without memory, there's no progress.")

The lesson is clear: **conversation history is what turns a stateless LLM call into a stateful agent**. Without it, the agent has amnesia — it repeats itself endlessly.

---
## 5. A Step-Counting Agent

Let's build an agent with very specific step-by-step behavior. We'll instruct it to count from 1 to 5, outputting exactly one number per step. This demonstrates precise control over the agent's behavior through the system prompt.

In [ ]:
COUNTING_PROMPT = """You are a counting agent. Your task is to count from 1 to 5.

Rules:
- Output EXACTLY one number per response.
- Just say the number and nothing else.
- When you reach 5, say FINAL ANSWER: done

Start counting now."""

messages = [
    {"role": "system", "content": COUNTING_PROMPT},
    {"role": "user", "content": "Begin."},
]

max_steps = 10  # generous limit — should finish in ~5

print("Step-Counting Agent")
print("=" * 30)

for step in range(1, max_steps + 1):
    try:
        response = chat(messages, temperature=0.0)
    except RuntimeError as e:
        print(f"Step {step}: LLM call failed: {e}")
        break

    print(f"Step {step}: {response.strip()}")

    if "FINAL ANSWER:" in response:
        final_answer = response.split("FINAL ANSWER:")[1].strip()
        print(f"\nAgent finished in {step} step(s). Result: {final_answer}")
        break

    messages.append({"role": "assistant", "content": response})
    messages.append({"role": "user", "content": "Next."})

    time.sleep(1)
else:
    print(f"\nAgent hit max steps ({max_steps}) without finishing.")

This is a powerful demonstration: the system prompt acts as the agent's **program**, and the loop is the **runtime**. The LLM follows the instructions step by step, outputting one number at a time, and knows to stop at 5.

Key observations:
- We set `temperature=0.0` for deterministic, predictable output
- The `user` message `"Next."` nudges the LLM to continue
- The `for...else` pattern lets us detect whether the loop finished naturally (timeout) or via `break` (success)

---
## Putting It Together: Multi-Step Research Agent

Now let's build something more realistic: a **multi-step research agent**. Given a topic, the agent will:

1. Generate 3 research questions about the topic
2. Answer each question one by one (one per step)
3. Synthesize a summary as the final answer

This demonstrates the full **observe-think-act** pattern that underlies all sophisticated agents.

In [ ]:
RESEARCH_PROMPT = """You are a research agent. Given a topic, you will research it step by step.

Follow this exact process:

STEP 1: Generate exactly 3 research questions about the topic. List them numbered.
STEP 2: Answer question 1 in 2-3 sentences.
STEP 3: Answer question 2 in 2-3 sentences.
STEP 4: Answer question 3 in 2-3 sentences.
STEP 5: Synthesize all your answers into a brief summary paragraph, then output:
FINAL ANSWER: <your summary>

Do exactly ONE step per response. Do not skip ahead.
Start each response with "STEP N:" where N is the current step number."""


def research_agent(topic: str, max_steps: int = 8) -> str:
    """Run a multi-step research agent on the given topic."""
    messages = [
        {"role": "system", "content": RESEARCH_PROMPT},
        {"role": "user", "content": f"Research this topic: {topic}"},
    ]

    print(f"Researching: {topic}")
    print("=" * 60)

    for step in range(1, max_steps + 1):
        print(f"\n--- Agent Step {step}/{max_steps} ---")

        try:
            response = chat(messages, temperature=0.4)
        except RuntimeError as e:
            print(f"LLM call failed: {e}")
            return f"Error: {e}"

        print(response)

        if "FINAL ANSWER:" in response:
            final_answer = response.split("FINAL ANSWER:")[1].strip()
            print("\n" + "=" * 60)
            print(f"Research complete in {step} step(s).")
            return final_answer

        messages.append({"role": "assistant", "content": response})
        messages.append({"role": "user", "content": "Proceed to the next step."})

        time.sleep(1)

    print(f"\nAgent hit max steps ({max_steps}).")
    return "Research incomplete — agent timed out."

In [ ]:
summary = research_agent("Why do programming languages have different paradigms?")
print(f"\nFinal Summary:\n{summary}")

This is the observe-think-act loop in action:

| Phase | What happens |
|-------|-------------|
| **Observe** | The LLM reads the conversation history (its prior steps) |
| **Think** | It determines which step comes next based on the protocol |
| **Act** | It generates the output for that step |

The system prompt defines the **plan**. The loop provides the **runtime**. The message history provides the **memory**. Together, they form an agent.

In the next notebook, we'll add **tools** — giving the agent the ability to actually *do* things (search, calculate, fetch data) instead of just reasoning from its own knowledge.

---
## Try It Yourself

Work through these exercises to deepen your understanding of the agent loop pattern.

### Exercise 1: Add a `max_steps` Safety Limit

Take the simple `while True` agent loop from Section 2 and add a `max_steps` safety limit. Test it with a deliberately open-ended question that might cause the agent to loop (e.g., `"Explain everything there is to know about physics."`). Verify that the agent stops after `max_steps` iterations even if it never says `FINAL ANSWER:`.

In [ ]:
# Exercise 1: Your code here
#
# Hints:
# - Use a `for` loop with `range(1, max_steps + 1)` instead of `while True`
# - After the loop, print a message indicating the agent timed out
# - Test with max_steps=3 and a hard question


### Exercise 2: Confidence-Based Stopping

Build an agent that stops when it's "confident" in its answer. The protocol:
- The system prompt tells the LLM to self-assess after each reasoning step
- It should output `CONFIDENCE: HIGH` when it's sure, followed by `FINAL ANSWER:`
- If confidence is `LOW` or `MEDIUM`, it should keep reasoning

Try it with a question like: `"Is the number 97 prime?"`

In [ ]:
# Exercise 2: Your code here
#
# Hints:
# - System prompt should say: "After each step, assess your confidence:
#   CONFIDENCE: LOW / MEDIUM / HIGH. When HIGH, give FINAL ANSWER:"
# - Check for both "CONFIDENCE: HIGH" and "FINAL ANSWER:" in the response
# - Don't forget max_steps as a safety net!


### Exercise 3: Step-by-Step Logging

Enhance the `run_agent` function from Section 3 to print detailed diagnostics at each step:
- Step number
- Number of messages in history
- Length of the LLM response (in characters)
- Elapsed time since the agent started

Use `time.time()` to track elapsed time.

In [ ]:
# Exercise 3: Your code here
#
# Hints:
# - Record start_time = time.time() before the loop
# - At each step: elapsed = time.time() - start_time
# - Print: f"Step {step} | Messages: {len(messages)} | Response: {len(response)} chars | Elapsed: {elapsed:.1f}s"


### Exercise 4 (Stretch): 20 Questions Agent

Build a "20 questions" agent:
1. You think of something (e.g., "elephant")
2. The agent asks yes/no questions to guess what it is
3. You answer each question using `input()` (type "yes" or "no")
4. When the agent is ready to guess, it says `FINAL ANSWER:` with its guess
5. Maximum 20 questions

This exercise demonstrates interactive agents where the user is part of the loop.

In [ ]:
# Exercise 4: Your code here
#
# Hints:
# - System prompt: "You are playing 20 questions. Ask one yes/no question per
#   response. When you're ready to guess, say FINAL ANSWER: <your guess>"
# - After each LLM response (that isn't a final answer), use:
#   user_input = input("Your answer (yes/no): ")
# - Append both the agent's question and the user's answer to messages
# - max_steps = 20


---

**Next up:** [03. Tool Use from Scratch](03_tool_use_from_scratch.ipynb) — giving the agent hands by adding tool calling to the loop.